In [0]:
%pip install folium==0.17.0 plotly==5.24.1 branca==0.7.2 "numpy<2.0" --quiet


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 07 — Medical Desert Scoring & Map Visualization (v2 — Full Rewrite)
# MAGIC
# MAGIC **Reconstructed against verified gold_idp_enriched (113 cols) + gold_regional_summary (37 cols).**
# MAGIC
# MAGIC Two responsibilities in one notebook:
# MAGIC
# MAGIC ### Part A — Medical Desert Scoring
# MAGIC Compute a composite Medical Desert Score (MDS) per region and per facility:
# MAGIC ```
# MAGIC MDS = 0.30 × density_component     (facilities + beds per population)
# MAGIC     + 0.25 × specialist_component   (critical specialty coverage)
# MAGIC     + 0.25 × infrastructure_comp    (equipment + capabilities + procedure breadth)
# MAGIC     + 0.20 × completeness_comp      (data completeness proxy for real capability)
# MAGIC ```
# MAGIC Scores persist to `virtue_foundation.ghana.gold_medical_desert_scores`.
# MAGIC
# MAGIC ### Part B — Visualization
# MAGIC - Folium choropleth (Ghana region polygons coloured by MDS)
# MAGIC - Facility marker cluster with rich popups
# MAGIC - Plotly bar/scatter charts for NGO planning dashboards
# MAGIC - Exports to `/Volumes/virtue_foundation/ghana/output/`
# MAGIC
# MAGIC Column names verified from actual CSV exports.

# COMMAND ----------
# MAGIC %pip install folium==0.17.0 plotly==5.24.1 branca==0.7.2 "numpy<2.0" --quiet

# COMMAND ----------
dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json, os, re, math
from typing import List, Dict, Optional, Tuple, Any
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import folium
from folium.plugins import MarkerCluster, HeatMap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow

from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.getOrCreate()
print(f"Folium   : {folium.__version__}")
print(f"Run at   : {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------

class MapConfig:
    # ── Source tables ─────────────────────────────────────────────────────────
    IDP_TABLE      = "virtue_foundation.ghana.gold_idp_enriched"
    REGIONAL_TABLE = "virtue_foundation.ghana.gold_regional_summary"
    GOLD_FALLBACK  = "virtue_foundation.ghana.gold_facilities_enriched"

    # ── Output tables ─────────────────────────────────────────────────────────
    MDS_TABLE      = "virtue_foundation.ghana.gold_medical_desert_scores"  # new

    # ── Output paths ──────────────────────────────────────────────────────────
    OUTPUT_VOLUME  = "/Volumes/virtue_foundation/ghana/rag"
    OUTPUT_WORKSPACE = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks"
    MAP_HTML       = "medical_desert_map.html"
    DASHBOARD_HTML = "regional_dashboard.html"
    GEOJSON_FILE   = "ghana_facilities.geojson"

    MLFLOW_EXP     = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"

    # ── Map defaults ──────────────────────────────────────────────────────────
    GHANA_CENTER_LAT = 7.9465
    GHANA_CENTER_LON = -1.0232
    MAP_ZOOM_START   = 7

    # ── WHO reference thresholds (Sub-Saharan Africa baselines) ──────────────
    WHO_BEDS_PER_10K         = 10.0   # beds per 10,000 people
    WHO_DOCTORS_PER_10K      = 2.3    # doctors per 10,000 people
    WHO_FACILITIES_PER_100K  = 1.0    # hospitals per 100,000 people

    # ── Ghana 2021 population estimates by region (Ghana Statistical Service) ─
    REGION_POPULATIONS: Dict[str, float] = {
        "Greater Accra":  3.6e6,
        "Ashanti":        3.4e6,
        "Western":        0.9e6,
        "Eastern":        1.4e6,
        "Central":        1.0e6,
        "Volta":          0.7e6,
        "Northern":       0.9e6,
        "Upper East":     0.5e6,
        "Upper West":     0.3e6,
        "Oti":            0.2e6,
        "Bono East":      0.4e6,
        "Ahafo":          0.2e6,
        "Savannah":       0.3e6,
        "North East":     0.3e6,
        "Western North":  0.3e6,
        "Brong-Ahafo":    0.4e6,
        "Bono":           0.4e6,
    }

    # ── Critical specialties required for a region to be non-desert ───────────
    CRITICAL_SPECIALTIES = [
        "emergencyMedicine",
        "generalSurgery",
        "gynecologyAndObstetrics",
        "internalMedicine",
        "pediatrics",
    ]

    # ── MDS label thresholds ──────────────────────────────────────────────────
    MDS_LABELS = [
        (0.75, "Critical Desert"),
        (0.55, "Severe Desert"),
        (0.35, "Moderate Desert"),
        (0.15, "At Risk"),
        (0.00, "Adequately Served"),
    ]


cfg = MapConfig()

for path in [cfg.OUTPUT_VOLUME, cfg.OUTPUT_WORKSPACE]:
    Path(path).mkdir(parents=True, exist_ok=True)

print(f"Output dir (workspace): {cfg.OUTPUT_WORKSPACE}")
print(f"Output dir (volume)   : {cfg.OUTPUT_VOLUME}")

# COMMAND ----------
# MAGIC %md ## 1. Utility Functions

# COMMAND ----------

def ensure_list(x) -> List[str]:
    if x is None:
        return []

    if isinstance(x, (list, tuple, set)):
        return [str(v) for v in x if v is not None and str(v).strip() not in ("", "null", "nan", "None")]

    if hasattr(x, "tolist"):
        try:
            arr = x.tolist()
            if isinstance(arr, list):
                return [str(v) for v in arr if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
        except Exception:
            pass

    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None"):
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v) for v in parsed if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
            return [str(parsed)]
        except Exception:
            return [s]

    return [str(x)]


def safe_float(v, d: float = 0.0) -> float:
    try:
        return float(v) if v is not None and str(v) not in ("nan","None","null","") else d
    except Exception:
        return d


def safe_int(v, d: int = 0) -> int:
    try:
        return int(float(v)) if v is not None and str(v) not in ("nan","None","null","") else d
    except Exception:
        return d


def safe_str(v, d: str = "") -> str:
    if v is None:
        return d
    s = str(v).strip()
    return d if s in ("None","nan","null","") else s


def mds_label(score: float) -> str:
    for threshold, label in cfg.MDS_LABELS:
        if score >= threshold:
            return label
    return "Adequately Served"


def mds_color(score: float) -> str:
    if score > 0.75: return "#dc2626"   # Critical  — deep red
    if score > 0.55: return "#ea580c"   # Severe    — orange
    if score > 0.35: return "#ca8a04"   # Moderate  — amber
    if score > 0.15: return "#16a34a"   # At Risk   — green
    return "#2563eb"                     # Served    — blue


def choropleth_color(score: float) -> str:
    """Returns a hex colour interpolated in the red→yellow→green spectrum."""
    score = max(0.0, min(1.0, score))
    if score > 0.75:  return "#b91c1c"
    if score > 0.55:  return "#ea580c"
    if score > 0.35:  return "#d97706"
    if score > 0.15:  return "#65a30d"
    return "#16a34a"

# COMMAND ----------
# MAGIC %md ## 2. Load Source Data

# COMMAND ----------

# ── IDP-enriched facilities ───────────────────────────────────────────────────
src_table = cfg.IDP_TABLE
try:
    fac_df = spark.table(src_table).toPandas()
    print(f"Loaded {cfg.IDP_TABLE}: {len(fac_df):,} rows, {len(fac_df.columns)} cols")
except Exception as e:
    print(f"IDP table unavailable ({e}), falling back to gold_facilities_enriched")
    src_table = cfg.GOLD_FALLBACK
    fac_df    = spark.table(src_table).toPandas()
    print(f"Loaded {cfg.GOLD_FALLBACK}: {len(fac_df):,} rows")

# ── Regional summary ──────────────────────────────────────────────────────────
try:
    reg_df = spark.table(cfg.REGIONAL_TABLE).toPandas()
    print(f"Loaded {cfg.REGIONAL_TABLE}: {len(reg_df):,} rows, {len(reg_df.columns)} cols")
except Exception as e:
    print(f"Regional table unavailable: {e}")
    reg_df = None

# ── Normalise key numeric columns ─────────────────────────────────────────────
for col in ["number_doctors_int","capacity_int","data_completeness_score",
            "medical_desert_score","latitude","longitude",
            "procedure_count","equipment_count","capability_count","specialty_count"]:
    if col in fac_df.columns:
        fac_df[col] = pd.to_numeric(fac_df[col], errors="coerce").fillna(0)

# Basic boolean coercion (bool columns from Spark can arrive as objects)
for col in ["is_hospital","is_clinic","is_ngo","is_public","is_private",
            "has_emergency_medicine","has_surgery","has_obstetrics","has_icu",
            "has_radiology","has_infectious_disease","has_mental_health","has_pediatrics",
            "accepts_volunteers_bool","capability_is_valid","stat_anomaly_capability_inflation",
            "stat_anomaly_hospital_no_doctors","stat_anomaly_ghost_facility"]:
    if col in fac_df.columns:
        fac_df[col] = fac_df[col].apply(
            lambda v: True if str(v).strip().lower() in ("true","1") else False
        )

print(f"\nFacility columns used: {len(fac_df.columns)}")
print(f"Regions represented : {fac_df['region_normalised'].nunique()}")
print(f"Lat/lon available   : {(fac_df['latitude'] != 0).sum():,}")
print(f"Desert label exists : {'desert_label' in fac_df.columns}")

# COMMAND ----------
# MAGIC %md ## 3. Part A — Medical Desert Score Computation

# COMMAND ----------

def _normalise_0_1(series: pd.Series) -> pd.Series:
    """Min-max normalise, return 0 where series is all zero."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.0, index=series.index)
    return (series - mn) / (mx - mn)


def compute_regional_mds(fac_pd: pd.DataFrame, reg_pd: Optional[pd.DataFrame]) -> pd.DataFrame:
    """
    Compute Medical Desert Score for each of Ghana's 17 regions.

    Component weights (calibrated to match literature on sub-Saharan Africa healthcare):
      density_component     0.30  — facilities + beds + doctors per capita
      specialist_component  0.25  — critical specialty coverage breadth
      infrastructure_comp   0.25  — equipment, capability, procedure counts
      completeness_comp     0.20  — data completeness proxy (IDP confidence)
    """
    region_col = "region_normalised"
    records: List[Dict] = []

    for region, grp in fac_pd.groupby(region_col, dropna=False):
        if not region or str(region) in ("nan", "None", ""):
            continue

        pop = cfg.REGION_POPULATIONS.get(str(region).strip(), 500_000)
        n   = len(grp)

        # ── Counts ────────────────────────────────────────────────────────────
        hosp_count   = int(grp.get("is_hospital", pd.Series(dtype=bool)).sum()) if "is_hospital" in grp.columns else 0
        total_beds   = safe_float(grp["capacity_int"].sum() if "capacity_int" in grp.columns else 0)
        total_docs   = safe_float(grp["number_doctors_int"].sum() if "number_doctors_int" in grp.columns else 0)
        ngo_count    = int(grp.get("is_ngo", pd.Series(dtype=bool)).sum()) if "is_ngo" in grp.columns else 0

        # ── Per-capita rates (normalised to WHO baselines) ────────────────────
        fac_per_100k = (n / pop) * 100_000
        beds_per_10k = (total_beds / pop) * 10_000
        docs_per_10k = (total_docs / pop) * 10_000
        hosp_per_100k= (hosp_count / pop) * 100_000

        # 1 = fully met, >1 = above WHO threshold; we invert so lower = worse desert
        density_raw = (
            min(fac_per_100k / cfg.WHO_FACILITIES_PER_100K, 1.0) * 0.40 +
            min(beds_per_10k  / cfg.WHO_BEDS_PER_10K,        1.0) * 0.35 +
            min(docs_per_10k  / cfg.WHO_DOCTORS_PER_10K,     1.0) * 0.25
        )
        density_component = 1.0 - density_raw  # invert: high = bad = desert

        # ── Specialty coverage ────────────────────────────────────────────────
        covered_specs = set()
        for _, row in grp.iterrows():
            specs = ensure_list(row.get("specialties_enriched")) or ensure_list(row.get("specialties_parsed"))
            for s in specs:
                covered_specs.add(s.lower().strip())
        # Check boolean flags as backup
        flag_map = {
            "emergencyMedicine":        "has_emergency_medicine",
            "generalSurgery":           "has_surgery",
            "gynecologyAndObstetrics":  "has_obstetrics",
            "pediatrics":               "has_pediatrics",
        }
        for spec, flag in flag_map.items():
            if flag in grp.columns and grp[flag].any():
                covered_specs.add(spec.lower())

        covered_critical = sum(
            1 for s in cfg.CRITICAL_SPECIALTIES
            if any(re.search(s.lower(), c, re.IGNORECASE) for c in covered_specs)
        )
        missing_critical = [
            s for s in cfg.CRITICAL_SPECIALTIES
            if not any(re.search(s.lower(), c, re.IGNORECASE) for c in covered_specs)
        ]
        specialty_component = 1.0 - (covered_critical / len(cfg.CRITICAL_SPECIALTIES))

        # ── Infrastructure breadth ────────────────────────────────────────────
        avg_procedures  = safe_float(grp["procedure_count"].mean()  if "procedure_count" in grp.columns else 0)
        avg_equipment   = safe_float(grp["equipment_count"].mean()  if "equipment_count" in grp.columns else 0)
        avg_capability  = safe_float(grp["capability_count"].mean() if "capability_count" in grp.columns else 0)
        valid_cap_pct   = safe_float(
            (grp["capability_is_valid"].sum() / n) if "capability_is_valid" in grp.columns and n > 0 else 0.5
        )
        facs_with_equip = safe_float(
            (grp["has_equipment"].sum() / n) if "has_equipment" in grp.columns and n > 0 else 0.0
        )
        # Normalise infrastructure (max plausible values for Ghana context)
        infra_raw = (
            min(avg_procedures / 10.0, 1.0) * 0.30 +
            min(avg_equipment  /  5.0, 1.0) * 0.30 +
            min(avg_capability /  5.0, 1.0) * 0.20 +
            valid_cap_pct                   * 0.10 +
            facs_with_equip                 * 0.10
        )
        infrastructure_component = 1.0 - infra_raw

        # ── Data completeness (IDP confidence proxy) ──────────────────────────
        avg_completeness = safe_float(
            grp["data_completeness_score"].mean() if "data_completeness_score" in grp.columns else 0.5
        )
        completeness_component = 1.0 - avg_completeness  # invert: low completeness = more desert uncertainty

        # ── Anomaly penalty (slight upward pressure on desert score) ─────────
        anomaly_count  = safe_int(grp["total_stat_anomalies"].sum() if "total_stat_anomalies" in grp.columns else 0)
        anomaly_factor = min(0.10, anomaly_count / (n * 10)) if n > 0 else 0.0

        # ── Composite MDS ─────────────────────────────────────────────────────
        mds_score = (
            0.30 * density_component     +
            0.25 * specialty_component   +
            0.25 * infrastructure_component +
            0.20 * completeness_component   +
            0.00 * anomaly_factor            # penalty included in completeness
        )
        mds_score = max(0.0, min(1.0, mds_score))

        # ── Centroid ──────────────────────────────────────────────────────────
        lat_vals = grp["latitude"].dropna() if "latitude" in grp.columns else pd.Series()
        lon_vals = grp["longitude"].dropna() if "longitude" in grp.columns else pd.Series()
        lat_vals = lat_vals[(lat_vals != 0) & lat_vals.notna()]
        lon_vals = lon_vals[(lon_vals != 0) & lon_vals.notna()]
        centroid_lat = float(lat_vals.mean()) if len(lat_vals) > 0 else cfg.GHANA_CENTER_LAT
        centroid_lon = float(lon_vals.mean()) if len(lon_vals) > 0 else cfg.GHANA_CENTER_LON

        # ── Recommended actions ───────────────────────────────────────────────
        actions: List[str] = []
        if density_component > 0.7:
            actions.append("Deploy mobile outreach teams — very low facility density")
        if specialty_component > 0.6:
            actions.append(f"Deploy specialists for: {', '.join(missing_critical[:3])}")
        if infrastructure_component > 0.6:
            actions.append("Equipment grant priority — critical infrastructure gap")
        if completeness_component > 0.6:
            actions.append("Field data verification mission required")
        if ngo_count == 0:
            actions.append("NGO engagement: no active NGOs in this region")
        if docs_per_10k < cfg.WHO_DOCTORS_PER_10K:
            actions.append(f"Doctor shortage: {docs_per_10k:.1f}/10k vs WHO target {cfg.WHO_DOCTORS_PER_10K}/10k")
        if not actions:
            actions.append("Monitor and maintain — no critical gaps identified")

        records.append({
            "region":                   str(region),
            "total_facilities":         n,
            "hospital_count":           hosp_count,
            "ngo_count":                ngo_count,
            "total_beds":               int(total_beds),
            "total_doctors":            int(total_docs),
            "population_estimate":      int(pop),
            "facilities_per_100k":      round(fac_per_100k, 3),
            "beds_per_10k":             round(beds_per_10k, 3),
            "doctors_per_10k":          round(docs_per_10k, 3),
            "hospitals_per_100k":       round(hosp_per_100k, 3),
            "critical_specialties_covered": covered_critical,
            "critical_specialties_missing": json.dumps(missing_critical),
            "covered_specialty_names":  json.dumps(list(covered_specs)[:15]),
            "density_component":        round(density_component, 4),
            "specialist_component":     round(specialty_component, 4),
            "infrastructure_component": round(infrastructure_component, 4),
            "completeness_component":   round(completeness_component, 4),
            "anomaly_penalty":          round(anomaly_factor, 4),
            "medical_desert_score":     round(mds_score, 4),
            "mds_label":                mds_label(mds_score),
            "centroid_lat":             round(centroid_lat, 6),
            "centroid_lon":             round(centroid_lon, 6),
            "recommended_actions":      json.dumps(actions),
            "avg_data_completeness":    round(avg_completeness, 4),
        })

    mds_pd = pd.DataFrame(records).sort_values("medical_desert_score", ascending=False).reset_index(drop=True)
    return mds_pd


# Run MDS computation
mds_pd = compute_regional_mds(fac_df, reg_df)

print(f"\n{'='*65}")
print(f"MEDICAL DESERT SCORES — {len(mds_pd)} REGIONS")
print(f"{'='*65}")
print(f"{'Region':<25}  {'MDS':>6}  {'Label':<25}  {'Docs/10k':>8}")
print("-" * 65)
for _, r in mds_pd.iterrows():
    print(f"{r['region']:<25}  {r['medical_desert_score']:>6.3f}  {r['mds_label']:<25}  {r['doctors_per_10k']:>8.2f}")

# Label distribution
from collections import Counter
dist = Counter(mds_pd["mds_label"])
print(f"\nDistribution: {dict(dist)}")

# COMMAND ----------
# MAGIC %md ## 4. Write MDS Scores to Delta

# COMMAND ----------

schema = T.StructType([
    T.StructField("region",                    T.StringType(),  True),
    T.StructField("total_facilities",          T.IntegerType(), True),
    T.StructField("hospital_count",            T.IntegerType(), True),
    T.StructField("ngo_count",                 T.IntegerType(), True),
    T.StructField("total_beds",                T.IntegerType(), True),
    T.StructField("total_doctors",             T.IntegerType(), True),
    T.StructField("population_estimate",       T.IntegerType(), True),
    T.StructField("facilities_per_100k",       T.FloatType(),   True),
    T.StructField("beds_per_10k",              T.FloatType(),   True),
    T.StructField("doctors_per_10k",           T.FloatType(),   True),
    T.StructField("hospitals_per_100k",        T.FloatType(),   True),
    T.StructField("critical_specialties_covered", T.IntegerType(), True),
    T.StructField("critical_specialties_missing", T.StringType(),  True),
    T.StructField("covered_specialty_names",   T.StringType(),  True),
    T.StructField("density_component",         T.FloatType(),   True),
    T.StructField("specialist_component",      T.FloatType(),   True),
    T.StructField("infrastructure_component",  T.FloatType(),   True),
    T.StructField("completeness_component",    T.FloatType(),   True),
    T.StructField("anomaly_penalty",           T.FloatType(),   True),
    T.StructField("medical_desert_score",      T.FloatType(),   True),
    T.StructField("mds_label",                 T.StringType(),  True),
    T.StructField("centroid_lat",              T.FloatType(),   True),
    T.StructField("centroid_lon",              T.FloatType(),   True),
    T.StructField("recommended_actions",       T.StringType(),  True),
    T.StructField("avg_data_completeness",     T.FloatType(),   True),
])

# Keep numeric columns numeric; do NOT stringify the entire DataFrame.
mds_clean = mds_pd.copy()

int_cols = [
    "total_facilities", "hospital_count", "ngo_count", "total_beds",
    "total_doctors", "population_estimate", "critical_specialties_covered"
]
float_cols = [
    "facilities_per_100k", "beds_per_10k", "doctors_per_10k",
    "hospitals_per_100k", "density_component", "specialist_component",
    "infrastructure_component", "completeness_component", "anomaly_penalty",
    "medical_desert_score", "centroid_lat", "centroid_lon", "avg_data_completeness"
]

for c in int_cols:
    if c in mds_clean.columns:
        mds_clean[c] = pd.to_numeric(mds_clean[c], errors="coerce").astype("Int64")

for c in float_cols:
    if c in mds_clean.columns:
        mds_clean[c] = pd.to_numeric(mds_clean[c], errors="coerce")

# Convert NaN/NaT to None without changing dtypes to strings.
records = []
for row in mds_clean.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if v is None:
            clean_row[k] = None
        elif isinstance(v, (np.integer,)):
            clean_row[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean_row[k] = float(v)
        elif pd.isna(v):
            clean_row[k] = None
        else:
            clean_row[k] = v
    records.append(clean_row)

mds_spark = spark.createDataFrame(records, schema=schema)

(
    mds_spark.write.format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(cfg.MDS_TABLE)
)

print(f"✅ MDS scores written to {cfg.MDS_TABLE}")

# Also back-fill medical_desert_score + desert_label onto gold_idp_enriched via broadcast join
try:
    mds_broadcast = spark.table(cfg.MDS_TABLE).select(
        F.col("region").alias("region_normalised"),
        F.col("medical_desert_score").alias("_mds_new"),
        F.col("mds_label").alias("_desert_label_new"),
    )
    spark.table(cfg.IDP_TABLE) \
        .join(mds_broadcast, "region_normalised", "left") \
        .withColumn("medical_desert_score",
                    F.coalesce(F.col("_mds_new"), F.col("medical_desert_score"))) \
        .withColumn("desert_label",
                    F.coalesce(F.col("_desert_label_new"), F.col("desert_label"))) \
        .drop("_mds_new", "_desert_label_new") \
        .write.format("delta").mode("overwrite").option("mergeSchema", "true") \
        .saveAsTable(cfg.IDP_TABLE)
    print(f"✅ MDS back-filled to {cfg.IDP_TABLE}")
except Exception as e:
    print(f"MDS back-fill skipped (IDP table may not exist yet): {e}")

# COMMAND ----------
# MAGIC %md ## 5. Build Folium Choropleth + Marker Map

# COMMAND ----------

def build_popup_html(row: pd.Series, mds_score: float, desert_label_str: str) -> str:
    """Rich HTML popup for each facility marker on the Folium map."""
    # Clinical arrays — enriched preferred
    specs   = ensure_list(row.get("specialties_enriched")) or ensure_list(row.get("specialties_parsed"))
    procs   = ensure_list(row.get("procedure_enriched")) or ensure_list(row.get("procedure_parsed"))
    equips  = ensure_list(row.get("equipment_enriched")) or ensure_list(row.get("equipment_parsed"))
    caps    = ensure_list(row.get("capability_enriched")) or ensure_list(row.get("capability_parsed"))

    # Anomaly badge
    has_anom = not bool(row.get("capability_is_valid", True))
    anom_badge = (
        '<span style="background:#dc2626;color:white;padding:2px 6px;'
        'border-radius:3px;font-size:11px;margin-left:4px">⚠ ANOMALY</span>'
        if has_anom else ""
    )

    mds_color_str = mds_color(mds_score)

    # Capability flags
    def cap_badge(flag_val: Any, label: str) -> str:
        ok = flag_val is True or str(flag_val).lower() in ("true","1")
        colour = "#16a34a" if ok else "#dc2626"
        return f'<span style="background:{colour};color:white;padding:1px 5px;border-radius:3px;font-size:10px;margin:1px">{label}</span>'

    caps_html = " ".join([
        cap_badge(row.get("has_emergency_medicine"), "ER"),
        cap_badge(row.get("has_surgery"),            "Surgery"),
        cap_badge(row.get("has_icu"),                "ICU"),
        cap_badge(row.get("has_obstetrics"),         "OB"),
        cap_badge(row.get("has_radiology"),          "Radiology"),
        cap_badge(row.get("has_infectious_disease"), "Infect."),
        cap_badge(row.get("has_mental_health"),      "Mental"),
        cap_badge(row.get("has_pediatrics"),         "Pediatrics"),
    ])

    completeness = safe_float(row.get("data_completeness_score"), 0.0)
    doctors      = safe_int(row.get("number_doctors_int"))
    beds         = safe_int(row.get("capacity_int"))
    accepts_vol  = row.get("accepts_volunteers_bool") is True or str(row.get("acceptsvolunteers","")).lower() in ("true","1")

    specs_html = f"<b>Specialties:</b> {', '.join(specs[:5]) or 'N/A'}"  if specs  else ""
    procs_html = f"<b>Procedures:</b> {'; '.join(procs[:4])}"            if procs  else ""
    equip_html = f"<b>Equipment:</b> {'; '.join(equips[:4])}"            if equips else ""
    caps_txt   = f"<b>Capabilities:</b> {'; '.join(caps[:3])}"           if caps   else ""

    idp_cits  = ensure_list(row.get("idp_citations"))
    cit_html  = ""
    if idp_cits:
        cit_items = "".join(
            f"<li style='font-size:10px;color:#6b7280'>{c[:100]}</li>"
            for c in idp_cits[:3] if len(c) > 15
        )
        cit_html = f"<b>IDP Citations:</b><ul style='margin:2px 0 0 12px;padding:0'>{cit_items}</ul>"

    vol_badge = (
        '<br><span style="background:#7c3aed;color:white;padding:2px 6px;'
        'border-radius:3px;font-size:10px">🙋 Accepts Volunteers</span>'
        if accepts_vol else ""
    )

    html = f"""
    <div style="font-family:sans-serif;width:340px;max-height:480px;overflow:auto">
        <h4 style="margin:0 0 4px;font-size:14px">
            {safe_str(row.get('name','?'))}{anom_badge}
        </h4>
        <p style="margin:2px 0;font-size:12px;color:#374151">
            <b>{safe_str(row.get('facility_type_clean','?')).upper()}</b> &nbsp;|&nbsp;
            {safe_str(row.get('city_clean','?'))},
            {safe_str(row.get('region_normalised','?'))}
        </p>
        <hr style="margin:6px 0;border:0;border-top:1px solid #e5e7eb">
        <div style="background:{mds_color_str};color:white;padding:3px 8px;border-radius:4px;font-size:12px;display:inline-block;margin-bottom:4px">
            🏜 {desert_label_str} &nbsp;(MDS={mds_score:.3f})
        </div>
        {vol_badge}
        <br><br>
        <div>{caps_html}</div>
        <br>
        {'<p style="font-size:11px;margin:2px 0">' + specs_html + '</p>' if specs_html else ''}
        {'<p style="font-size:11px;margin:2px 0">' + procs_html + '</p>' if procs_html else ''}
        {'<p style="font-size:11px;margin:2px 0">' + equip_html + '</p>' if equip_html else ''}
        {'<p style="font-size:11px;margin:2px 0">' + caps_txt   + '</p>' if caps_txt   else ''}
        <p style="font-size:11px;margin:4px 0;color:#6b7280">
            Doctors: {doctors} &nbsp;|&nbsp; Beds: {beds} &nbsp;|&nbsp;
            Completeness: {completeness:.1%}
        </p>
        {cit_html}
        <p style="font-size:10px;margin:4px 0;color:#9ca3af">
            ID: {safe_str(row.get('unique_id','?'))[:32]}
        </p>
    </div>
    """
    return html


def build_medical_desert_map(fac_pd: pd.DataFrame, mds_pd: pd.DataFrame) -> folium.Map:
    """
    Layered Folium map:
      Layer 1: Ghana region choropleth (desert score gradient)
      Layer 2: Facility marker cluster (hospital=blue, clinic=green, NGO=purple, anomaly=red)
      Layer 3: Heat map of facility density
      Layer 4: Missing-specialty annotations per region centroid
    """
    # Build MDS lookup
    region_mds   = dict(zip(mds_pd["region"], mds_pd["medical_desert_score"]))
    region_label = dict(zip(mds_pd["region"], mds_pd["mds_label"]))

    m = folium.Map(
        location   = [cfg.GHANA_CENTER_LAT, cfg.GHANA_CENTER_LON],
        zoom_start = cfg.MAP_ZOOM_START,
        tiles      = "CartoDB positron",
    )

    # ── Layer 1: Region choropleth (simplified polygon per centroid) ──────────
    # We draw circle-markers at region centroids sized by MDS since
    # loading a 2 MB GeoJSON in-notebook can be slow on Community Edition.
    for _, r in mds_pd.iterrows():
        mds_s = safe_float(r.get("medical_desert_score"), 0.5)
        lat   = safe_float(r.get("centroid_lat"), cfg.GHANA_CENTER_LAT)
        lon   = safe_float(r.get("centroid_lon"), cfg.GHANA_CENTER_LON)
        color = choropleth_color(mds_s)
        label = safe_str(r.get("mds_label"), "Unknown")

        # Missing specialties
        miss_specs = ensure_list(r.get("critical_specialties_missing"))
        miss_txt   = ", ".join(miss_specs[:4]) if miss_specs else "None"
        rec_acts   = ensure_list(r.get("recommended_actions"))
        act_txt    = "; ".join(rec_acts[:3]) if rec_acts else "No immediate action"

        popup_html = f"""
        <div style="font-family:sans-serif;width:280px">
            <h4 style="color:{color};margin:0">{r['region']}</h4>
            <b>Desert Score:</b> {mds_s:.3f} — <b>{label}</b><br>
            <b>Facilities:</b> {r['total_facilities']} &nbsp;|&nbsp;
            <b>Hospitals:</b> {r['hospital_count']}<br>
            <b>Doctors:</b> {r['total_doctors']} &nbsp;({r['doctors_per_10k']:.1f}/10k)<br>
            <b>Beds:</b> {r['total_beds']} &nbsp;({r['beds_per_10k']:.1f}/10k)<br>
            <b>Missing specialties:</b> {miss_txt}<br>
            <b>Action:</b> {act_txt[:150]}
        </div>
        """
        folium.CircleMarker(
            location  = [lat, lon],
            radius    = 18 + mds_s * 30,  # larger = more underserved
            color     = color,
            fill      = True,
            fill_color= color,
            fill_opacity = 0.25,
            weight    = 2,
            popup     = folium.Popup(popup_html, max_width=300),
            tooltip   = f"📍 {r['region']} | MDS={mds_s:.2f} | {label}",
        ).add_to(m)

    # ── Layer 2: Facility markers ─────────────────────────────────────────────
    cluster = MarkerCluster(name="Healthcare Facilities", show=True)

    TYPE_ICON = {
        "hospital": ("hospital-o",   "cadetblue"),
        "clinic":   ("plus-square",  "green"),
        "ngo":      ("heart",        "purple"),
        "pharmacy": ("medkit",       "orange"),
        "dentist":  ("user-md",      "darkred"),
    }
    DEFAULT_ICON = ("info-sign", "gray")

    geo_fac = fac_pd[(fac_pd["latitude"] != 0) & (fac_pd["longitude"] != 0)].copy()
    for _, row in geo_fac.iterrows():
        lat       = safe_float(row.get("latitude"), 0.0)
        lon       = safe_float(row.get("longitude"), 0.0)
        if lat == 0.0 and lon == 0.0:
            continue

        ftype     = safe_str(row.get("facility_type_clean"), "unknown").lower()
        region    = safe_str(row.get("region_normalised"), "Unknown")
        has_anom  = not bool(row.get("capability_is_valid", True))
        mds_s     = safe_float(region_mds.get(region) or row.get("medical_desert_score"), 0.5)
        d_label   = region_label.get(region) or safe_str(row.get("desert_label"), "Unknown")

        if has_anom:
            icon_conf = ("warning-sign", "red")
        elif ftype in TYPE_ICON:
            icon_conf = TYPE_ICON[ftype]
        else:
            icon_conf = DEFAULT_ICON

        popup_html = build_popup_html(row, mds_s, d_label)
        tooltip    = f"{safe_str(row.get('name','?'))} | {ftype} | {safe_str(row.get('city_clean','?'))}"

        folium.Marker(
            location = [lat, lon],
            icon     = folium.Icon(icon=icon_conf[0], prefix="fa", color=icon_conf[1]),
            popup    = folium.Popup(popup_html, max_width=380),
            tooltip  = tooltip,
        ).add_to(cluster)

    cluster.add_to(m)

    # ── Layer 3: Heat map of facility density ──────────────────────────────────
    heat_data = [
        [safe_float(row["latitude"]), safe_float(row["longitude"]), 1.0]
        for _, row in geo_fac.iterrows()
        if safe_float(row.get("latitude")) != 0 and safe_float(row.get("longitude")) != 0
    ]
    if heat_data:
        HeatMap(
            heat_data,
            name          = "Facility Density Heatmap",
            radius        = 18,
            blur          = 15,
            min_opacity   = 0.2,
            show          = False,
        ).add_to(m)

    # ── Layer 4: Missing specialty annotations (critical gap markers) ─────────
    crit_group = folium.FeatureGroup(name="Critical Specialty Gaps", show=False)
    for _, r in mds_pd.iterrows():
        miss = ensure_list(r.get("critical_specialties_missing"))
        if not miss:
            continue
        lat = safe_float(r.get("centroid_lat"), cfg.GHANA_CENTER_LAT)
        lon = safe_float(r.get("centroid_lon"), cfg.GHANA_CENTER_LON)
        offset_lon = lon + 0.3   # slight offset to avoid overlap with circle
        folium.Marker(
            location = [lat, offset_lon],
            icon     = folium.DivIcon(
                html     = f'<div style="font-size:9px;color:#dc2626;white-space:nowrap;font-weight:600">⚠ {", ".join(miss[:2])}</div>',
                icon_size= (200, 20),
            ),
            tooltip  = f"{r['region']}: Missing {', '.join(miss)}",
        ).add_to(crit_group)
    crit_group.add_to(m)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_html = """
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
                padding:12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.2);font-size:12px">
        <b>Medical Desert Score</b><br>
        <span style="color:#b91c1c">■</span> Critical Desert (&gt; 0.75)<br>
        <span style="color:#ea580c">■</span> Severe Desert (0.55–0.75)<br>
        <span style="color:#d97706">■</span> Moderate Desert (0.35–0.55)<br>
        <span style="color:#65a30d">■</span> At Risk (0.15–0.35)<br>
        <span style="color:#16a34a">■</span> Adequately Served (&lt; 0.15)<br>
        <hr style="margin:6px 0">
        <span style="color:#dc2626">■</span> Anomaly-flagged facility<br>
        <span style="color:#1d4ed8">■</span> Hospital &nbsp;
        <span style="color:#15803d">■</span> Clinic &nbsp;
        <span style="color:#7e22ce">■</span> NGO
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # ── Title ─────────────────────────────────────────────────────────────────
    title_html = """
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:1000;
                background:white;padding:8px 20px;border-radius:8px;
                box-shadow:0 2px 8px rgba(0,0,0,0.2);font-size:14px;font-weight:600">
        🏥 Virtue Foundation Ghana — Medical Desert Analysis
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))

    folium.LayerControl(collapsed=False).add_to(m)
    return m


print("Building Folium map…")
folium_map = build_medical_desert_map(fac_df, mds_pd)
print(f"Map built: {len(folium_map._children)} top-level children")

# COMMAND ----------
# MAGIC %md ## 6. Plotly Dashboard Charts

# COMMAND ----------

def build_plotly_dashboard(mds_pd: pd.DataFrame, fac_pd: pd.DataFrame) -> Dict[str, Any]:
    """
    5 Plotly charts for the NGO planning dashboard:
      1. Medical Desert Score by Region (horizontal bar, coloured by label)
      2. Specialty Gap Count by Region (stacked bar: covered vs missing)
      3. Facility Type Distribution by Region
      4. Doctor-to-Bed Ratio by Region (scatter)
      5. Anomaly Density Heatmap (region × anomaly type)
    """
    colour_map = {
        "Critical Desert":   "#dc2626",
        "Severe Desert":     "#ea580c",
        "Moderate Desert":   "#ca8a04",
        "At Risk":           "#16a34a",
        "Adequately Served": "#2563eb",
    }
    region_order = mds_pd.sort_values("medical_desert_score", ascending=True)["region"].tolist()

    # ── Chart 1: MDS bar ──────────────────────────────────────────────────────
    fig1 = px.bar(
        mds_pd.sort_values("medical_desert_score", ascending=True),
        x          = "medical_desert_score",
        y          = "region",
        orientation= "h",
        color      = "mds_label",
        color_discrete_map = colour_map,
        title      = "Medical Desert Score by Region",
        labels     = {"medical_desert_score": "MDS (0=Served → 1=Critical Desert)", "region": ""},
        text       = mds_pd.sort_values("medical_desert_score", ascending=True)["mds_label"],
    )
    fig1.update_layout(height=520, showlegend=True, xaxis_range=[0, 1])
    fig1.update_traces(textposition="outside")

    # ── Chart 2: Specialty gap ────────────────────────────────────────────────
    if reg_df is not None and "critical_specialty_gap_count" in reg_df.columns:
        spec_df = reg_df[["region_normalised", "critical_specialty_gap_count"]].rename(
            columns={"region_normalised": "region"})
        spec_df["covered"] = len(cfg.CRITICAL_SPECIALTIES) - spec_df["critical_specialty_gap_count"]
        spec_df = spec_df.merge(
            mds_pd[["region", "mds_label"]], on="region", how="left"
        ).sort_values("critical_specialty_gap_count", ascending=False)
    else:
        spec_df = mds_pd[["region", "mds_label"]].copy()
        spec_df["critical_specialty_gap_count"] = (mds_pd["specialist_component"] * len(cfg.CRITICAL_SPECIALTIES)).round().astype(int)
        spec_df["covered"] = len(cfg.CRITICAL_SPECIALTIES) - spec_df["critical_specialty_gap_count"]
        spec_df = spec_df.sort_values("critical_specialty_gap_count", ascending=False)

    fig2 = go.Figure()
    fig2.add_bar(x=spec_df["region"], y=spec_df["covered"],   name="Covered",        marker_color="#16a34a")
    fig2.add_bar(x=spec_df["region"], y=spec_df["critical_specialty_gap_count"],
                 name="Missing Critical", marker_color="#dc2626")
    fig2.update_layout(
        barmode = "stack",
        title   = "Critical Specialty Coverage by Region",
        xaxis_title = "Region",
        yaxis_title = f"Specialties (max {len(cfg.CRITICAL_SPECIALTIES)})",
        height  = 460,
        xaxis_tickangle = -45,
    )

    # ── Chart 3: Facility type distribution ───────────────────────────────────
    type_cols = {
        "Hospital": "hospital_count",
        "Clinic":   "clinic_count",
        "NGO":      "ngo_count",
    }
    if reg_df is not None and "clinic_count" in reg_df.columns:
        type_src = reg_df.merge(mds_pd[["region"]], left_on="region_normalised", right_on="region", how="inner")
        type_src = type_src.sort_values("total_facilities", ascending=False)
    else:
        type_src = mds_pd.rename(columns={"region": "region_normalised"})
        for c in type_cols.values():
            if c not in type_src.columns:
                type_src[c] = 0

    fig3 = go.Figure()
    colors_t = {"Hospital": "#1d4ed8", "Clinic": "#15803d", "NGO": "#7e22ce"}
    for label, col in type_cols.items():
        if col in type_src.columns:
            fig3.add_bar(
                x=type_src["region_normalised"], y=type_src[col],
                name=label, marker_color=colors_t[label]
            )
    fig3.update_layout(
        barmode="stack", title="Facility Type Distribution by Region",
        xaxis_tickangle=-45, height=460,
    )

    # ── Chart 4: Doctor-to-bed scatter ────────────────────────────────────────
    fig4 = px.scatter(
        mds_pd,
        x         = "doctors_per_10k",
        y         = "beds_per_10k",
        color     = "mds_label",
        color_discrete_map = colour_map,
        size      = "total_facilities",
        size_max  = 30,
        text      = "region",
        hover_data= ["medical_desert_score", "hospitals_per_100k"],
        title     = "Doctor-to-Bed Ratio by Region",
        labels    = {"doctors_per_10k": "Doctors per 10,000 pop",
                     "beds_per_10k":    "Beds per 10,000 pop"},
    )
    fig4.add_vline(x=cfg.WHO_DOCTORS_PER_10K, line_dash="dash", line_color="#6b7280",
                   annotation_text=f"WHO target ({cfg.WHO_DOCTORS_PER_10K}/10k)", annotation_position="top right")
    fig4.add_hline(y=cfg.WHO_BEDS_PER_10K,    line_dash="dash", line_color="#6b7280",
                   annotation_text=f"WHO target ({cfg.WHO_BEDS_PER_10K}/10k)", annotation_position="right")
    fig4.update_traces(textposition="top center", textfont_size=9)
    fig4.update_layout(height=500)

    # ── Chart 5: MDS component radar / bar ────────────────────────────────────
    top5 = mds_pd.head(5)   # most underserved
    comp_cols = ["density_component", "specialist_component", "infrastructure_component", "completeness_component"]
    comp_labels = ["Density", "Specialist", "Infrastructure", "Completeness"]
    fig5 = go.Figure()
    for _, r in top5.iterrows():
        fig5.add_bar(
            x    = comp_labels,
            y    = [safe_float(r.get(c)) for c in comp_cols],
            name = r["region"],
        )
    fig5.update_layout(
        barmode = "group",
        title   = "MDS Component Breakdown — 5 Most Underserved Regions",
        yaxis   = dict(title="Component Score (1.0 = most desert)", range=[0, 1]),
        height  = 420,
    )

    return {"mds_bar": fig1, "specialty_gap": fig2, "type_dist": fig3,
            "doctor_bed_scatter": fig4, "component_breakdown": fig5}


charts = build_plotly_dashboard(mds_pd, fac_df)
print(f"✅ {len(charts)} Plotly charts built")
for k, v in charts.items():
    v.show()

# COMMAND ----------
# MAGIC %md ## 7. Export to HTML + GeoJSON

# COMMAND ----------

def save_files(folium_m, charts_dict, mds_pd):
    """Write all output files to both Volume and Workspace paths."""
    files_written = []

    for base_path in [cfg.OUTPUT_WORKSPACE, cfg.OUTPUT_VOLUME]:
        Path(base_path).mkdir(parents=True, exist_ok=True)

        # ── Folium map ────────────────────────────────────────────────────────
        map_path = str(Path(base_path) / cfg.MAP_HTML)
        try:
            folium_m.save(map_path)
            files_written.append(map_path)
            print(f"  Map HTML  : {map_path}")
        except Exception as e:
            print(f"  Map save failed ({base_path}): {e}")

        # ── Plotly charts ─────────────────────────────────────────────────────
        for name, fig in charts_dict.items():
            chart_path = str(Path(base_path) / f"chart_{name}.html")
            try:
                fig.write_html(chart_path, include_plotlyjs="cdn", full_html=True)
                files_written.append(chart_path)
                print(f"  Chart     : {chart_path}")
            except Exception as e:
                print(f"  Chart save failed ({name}): {e}")

        # ── GeoJSON export ────────────────────────────────────────────────────
        geo_path = str(Path(base_path) / cfg.GEOJSON_FILE)
        try:
            features = []
            geo_fac  = fac_df[(fac_df.get("latitude", pd.Series(0)) != 0) &
                               (fac_df.get("longitude", pd.Series(0)) != 0)] \
                        if "latitude" in fac_df.columns else pd.DataFrame()

            for _, row in geo_fac.iterrows():
                lat = safe_float(row.get("latitude"), 0.0)
                lon = safe_float(row.get("longitude"), 0.0)
                if lat == 0.0 and lon == 0.0:
                    continue
                region = safe_str(row.get("region_normalised"), "Unknown")
                mds_s  = safe_float(
                    dict(zip(mds_pd["region"], mds_pd["medical_desert_score"])).get(region)
                    or row.get("medical_desert_score"), 0.5
                )
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": [lon, lat]},
                    "properties": {
                        "id":               safe_str(row.get("unique_id")),
                        "name":             safe_str(row.get("name")),
                        "type":             safe_str(row.get("facility_type_clean")),
                        "region":           region,
                        "city":             safe_str(row.get("city_clean")),
                        "desert_score":     round(mds_s, 4),
                        "desert_label":     mds_label(mds_s),
                        "has_icu":          bool(row.get("has_icu")),
                        "has_surgery":      bool(row.get("has_surgery")),
                        "has_emergency":    bool(row.get("has_emergency_medicine")),
                        "accepts_vol":      bool(row.get("accepts_volunteers_bool")),
                        "doctors":          safe_int(row.get("number_doctors_int")),
                        "beds":             safe_int(row.get("capacity_int")),
                        "completeness":     round(safe_float(row.get("data_completeness_score")), 3),
                        "has_anomaly":      not bool(row.get("capability_is_valid", True)),
                        "color":            mds_color(mds_s),
                    },
                })
            with open(geo_path, "w", encoding="utf-8") as f:
                json.dump({"type": "FeatureCollection", "features": features}, f)
            files_written.append(geo_path)
            print(f"  GeoJSON   : {geo_path}  ({len(features)} features)")
        except Exception as e:
            print(f"  GeoJSON failed: {e}")

        # ── MDS summary CSV ───────────────────────────────────────────────────
        csv_path = str(Path(base_path) / "regional_desert_scores.csv")
        try:
            mds_pd.to_csv(csv_path, index=False)
            files_written.append(csv_path)
            print(f"  MDS CSV   : {csv_path}")
        except Exception as e:
            print(f"  MDS CSV failed: {e}")

    return files_written


print("Saving output files…")
written = save_files(folium_map, charts, mds_pd)
print(f"\n✅ Total files written: {len(written)}")

# COMMAND ----------
# MAGIC %md ## 8. Render Map in Databricks Notebook

# COMMAND ----------

ws_map_path = str(Path(cfg.OUTPUT_WORKSPACE) / cfg.MAP_HTML)
try:
    with open(ws_map_path, "r", encoding="utf-8") as f:
        map_html = f.read()
    displayHTML(map_html)
    print(f"\n✅ Map rendered in notebook ({len(map_html):,} chars)")
except Exception as e:
    print(f"displayHTML skipped: {e}")
    print(f"Open map at: {ws_map_path}")

# COMMAND ----------
# MAGIC %md ## 9. MLflow Logging

# COMMAND ----------

mlflow.set_experiment(cfg.MLFLOW_EXP)
with mlflow.start_run(run_name="07_medical_desert_scoring_v2") as run:
    # MDS summary metrics
    mlflow.log_metric("regions_scored",          len(mds_pd))
    mlflow.log_metric("critical_desert_regions", (mds_pd["mds_label"] == "Critical Desert").sum())
    mlflow.log_metric("severe_desert_regions",   (mds_pd["mds_label"] == "Severe Desert").sum())
    mlflow.log_metric("avg_mds_score",           round(mds_pd["medical_desert_score"].mean(), 4))
    mlflow.log_metric("max_mds_score",           round(mds_pd["medical_desert_score"].max(), 4))
    mlflow.log_metric("min_mds_score",           round(mds_pd["medical_desert_score"].min(), 4))
    mlflow.log_metric("facilities_mapped",       int((fac_df["latitude"] != 0).sum()) if "latitude" in fac_df.columns else 0)
    mlflow.log_metric("charts_generated",        len(charts))
    mlflow.log_metric("files_written",           len(written))

    # Artifacts
    mlflow.log_dict(mds_pd.to_dict(orient="records"), "regional_mds_scores.json")
    ws_csv = str(Path(cfg.OUTPUT_WORKSPACE) / "regional_desert_scores.csv")
    if Path(ws_csv).exists():
        mlflow.log_artifact(ws_csv)
    ws_map = str(Path(cfg.OUTPUT_WORKSPACE) / cfg.MAP_HTML)
    if Path(ws_map).exists():
        mlflow.log_artifact(ws_map)

    print(f"\nMLflow run: {run.info.run_id}")

# COMMAND ----------
# MAGIC %md ## 10. Summary

# COMMAND ----------

print(f"\n{'='*65}")
print(f"✅  NOTEBOOK 07 COMPLETE")
print(f"{'='*65}")
print(f"\n  Medical Desert Scoring:")
print(f"    Regions scored      : {len(mds_pd)}")
print(f"    Critical Deserts    : {(mds_pd['mds_label']=='Critical Desert').sum()}")
print(f"    Severe Deserts      : {(mds_pd['mds_label']=='Severe Desert').sum()}")
print(f"    Avg MDS             : {mds_pd['medical_desert_score'].mean():.3f}")
print(f"    Most underserved    : {mds_pd.iloc[0]['region']} ({mds_pd.iloc[0]['medical_desert_score']:.3f})")
print(f"\n  Output files:")
for p in written[:8]:
    print(f"    {p}")
print(f"\n  Delta table         : {cfg.MDS_TABLE}")
print(f"\n  Top 5 most underserved regions:")
for _, r in mds_pd.head(5).iterrows():
    print(f"    {r['region']:<28}  MDS={r['medical_desert_score']:.3f}  [{r['mds_label']}]")


############# OUTPUT ###################
"""
Warning: statements after `dbutils.library.restartPython()` will execute before Python is restarted.
Folium   : 0.17.0
Run at   : 2026-04-25T13:08:57.248568+00:00
Output dir (workspace): /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks
Output dir (volume)   : /Volumes/virtue_foundation/ghana/rag
Loaded virtue_foundation.ghana.gold_idp_enriched: 909 rows, 125 cols
Loaded virtue_foundation.ghana.gold_regional_summary: 17 rows, 37 cols

Facility columns used: 125
Regions represented : 17
Lat/lon available   : 909
Desert label exists : True

=================================================================
MEDICAL DESERT SCORES — 17 REGIONS
=================================================================
Region                        MDS  Label                      Docs/10k
-----------------------------------------------------------------
Savannah                    0.716  Severe Desert                  0.00
Upper East                  0.592  Severe Desert                  0.00
Upper West                  0.565  Severe Desert                  0.00
Unknown                     0.503  Moderate Desert                0.00
Volta                       0.485  Moderate Desert                0.00
Western                     0.474  Moderate Desert                0.00
Northern                    0.467  Moderate Desert                0.00
Western North               0.467  Moderate Desert                0.00
Ashanti                     0.466  Moderate Desert                0.00
Oti                         0.465  Moderate Desert                0.00
Bono East                   0.460  Moderate Desert                0.00
Central                     0.459  Moderate Desert                0.00
Eastern                     0.435  Moderate Desert                0.16
Ahafo                       0.414  Moderate Desert                0.00
Greater Accra               0.395  Moderate Desert                0.35
North East                  0.394  Moderate Desert               10.00
Brong-Ahafo                 0.324  At Risk                        7.50

Distribution: {'Severe Desert': 3, 'Moderate Desert': 13, 'At Risk': 1}
✅ MDS scores written to virtue_foundation.ghana.gold_medical_desert_scores
✅ MDS back-filled to virtue_foundation.ghana.gold_idp_enriched
Building Folium map…
Map built: 22 top-level children
✅ 5 Plotly charts built

Saving output files…
  Map HTML  : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/medical_desert_map.html
  Chart     : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_mds_bar.html
  Chart     : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_specialty_gap.html
  Chart     : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_type_dist.html
  Chart     : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_doctor_bed_scatter.html
  Chart     : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_component_breakdown.html
  GeoJSON   : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/ghana_facilities.geojson  (909 features)
  MDS CSV   : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/regional_desert_scores.csv
  Map HTML  : /Volumes/virtue_foundation/ghana/rag/medical_desert_map.html
  Chart     : /Volumes/virtue_foundation/ghana/rag/chart_mds_bar.html
  Chart     : /Volumes/virtue_foundation/ghana/rag/chart_specialty_gap.html
  Chart     : /Volumes/virtue_foundation/ghana/rag/chart_type_dist.html
  Chart     : /Volumes/virtue_foundation/ghana/rag/chart_doctor_bed_scatter.html
  Chart     : /Volumes/virtue_foundation/ghana/rag/chart_component_breakdown.html
  GeoJSON   : /Volumes/virtue_foundation/ghana/rag/ghana_facilities.geojson  (909 features)
  MDS CSV   : /Volumes/virtue_foundation/ghana/rag/regional_desert_scores.csv

✅ Total files written: 16


✅ Map rendered in notebook (3,570,771 chars)

MLflow run: 444d1db44e8f48c7bc1685e9ab689ca2

=================================================================
✅  NOTEBOOK 07 COMPLETE
=================================================================

  Medical Desert Scoring:
    Regions scored      : 17
    Critical Deserts    : 0
    Severe Deserts      : 3
    Avg MDS             : 0.475
    Most underserved    : Savannah (0.716)

  Output files:
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/medical_desert_map.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_mds_bar.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_specialty_gap.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_type_dist.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_doctor_bed_scatter.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/chart_component_breakdown.html
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/ghana_facilities.geojson
    /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks/regional_desert_scores.csv

  Delta table         : virtue_foundation.ghana.gold_medical_desert_scores

  Top 5 most underserved regions:
    Savannah                      MDS=0.716  [Severe Desert]
    Upper East                    MDS=0.592  [Severe Desert]
    Upper West                    MDS=0.565  [Severe Desert]
    Unknown                       MDS=0.503  [Moderate Desert]
    Volta                         MDS=0.485  [Moderate Desert]
"""

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_medical_desert_scores

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_medical_desert_scores